# RoPE band ablation — long-context extension (8k / 16k)

Tests the finding from the 1k–4k run: *a band starts to matter once its wavelength is comparable to the context length* (band 4, λ 6k–28k, became important for vt/fwe only at 4096). At 8k/16k it should strengthen, and band 5 (λ 35k–160k) may start to matter.

Same model, same factor (32, from calibration), same seeds. Tasks: s_niah, vt, fwe (mk_niah dropped — it tracked s_niah exactly; spec §7 cut order). Results from the 1k–4k run come with the repo and are skipped.

**Run one length per commit** (≈4 h for 8192, ≈8–9 h for 16384 on T4): set `LONG_LENGTH`, then *Save Version → Save & Run All*. Settings: GPU T4 x2, Internet On, secret `GITHUB_TOKEN`. To keep the 8192 results when running 16384, attach the 8192 version's output as input (cell 2 restores it) — or send the zip to be committed to the repo.

In [ ]:
# 1. Setup — edit these
LONG_LENGTH = 8192        # session A: 8192, session B: 16384
TASKS = ["s_niah", "vt", "fwe"]
MIN_BASELINE = 0.2        # a task with lower baseline at this length carries no signal -> skipped in the sweep
REPO_URL = "https://github.com/leeduy0403/rope-freq-ablation.git"
PRIVATE_REPO = True
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

import os, subprocess, glob, json, shutil

def sh(cmd):
    """Run a shell command, stream its output, raise if it fails (stops Run All)."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")

url = REPO_URL
if PRIVATE_REPO:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
REPO = "/kaggle/working/rope-freq-ablation"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q", url, "main"], check=True)
else:
    subprocess.run(["git", "clone", "-q", url, REPO], check=True)
os.chdir(REPO)
sh("git log --oneline -1 && pip install -q pyyaml")
M = f"--model-id {MODEL_ID}"
slug = MODEL_ID.replace("/", "__")


In [ ]:
# 2. Restore results from a previous version (if attached as input)
for src in glob.glob("/kaggle/input/*/rope-freq-ablation/results") + glob.glob("/kaggle/input/*/results"):
    print("restoring from", src)
    shutil.copytree(src, "results", dirs_exist_ok=True)
print("existing sweep cells:", len(glob.glob(f"results/{slug}/sweep/compress_f32/*.json")))


In [ ]:
# Environment + unit tests
sh("python -m src.env_report")
sh("python -m pytest -q tests")


In [ ]:
# Baseline at LONG_LENGTH (G2 thresholds are defined for the shortest length, so not enforced here)
sh(f"python -m src.eval.runner --stage baseline {M} --tasks {' '.join(TASKS)} --lengths {LONG_LENGTH} --no-gate")
base = {t: json.load(open(f"results/{slug}/baseline/{t}_{LONG_LENGTH}.json"))["accuracy"] for t in TASKS}
SWEEP_TASKS = [t for t in TASKS if base[t] >= MIN_BASELINE]
for t in TASKS:
    print(f"{t}@{LONG_LENGTH}: baseline {base[t]:.2f}", "" if t in SWEEP_TASKS else "-> SKIPPED (no signal)")
if not SWEEP_TASKS:
    raise RuntimeError("no task has signal at this length — stop and report")


In [ ]:
# Sweep at LONG_LENGTH, factor from config (32). Gate G4 enforced (band=none must reproduce baseline).
sh(f"python -m src.eval.runner --stage sweep {M} --tasks {' '.join(SWEEP_TASKS)} --lengths {LONG_LENGTH}")


In [ ]:
# Analysis over every length available (1k-4k from the repo + this run)
sh(f"python -m src.analyze.heatmap {M}")
sh(f"python -m src.analyze.overlay {M} --scale 4")
from IPython.display import Image, display
for p in sorted(glob.glob("figures/*.png")):
    print(p); display(Image(p))


In [ ]:
# Package results for download
sh("cd /kaggle/working && zip -qr results.zip rope-freq-ablation/results rope-freq-ablation/figures && ls -la results.zip")
